# SEP_CFE_DiCE Quickstart\n\nThis notebook demonstrates a complete constrained-counterfactual workflow using the standalone `sep-cfe-dice` package.

## 1) Install package (editable)\n\nRun this once if you opened the notebook from `examples/`.

In [ ]:
%pip install -e ".."

## 2) Imports

In [ ]:
import numpy as np\nimport pandas as pd\nfrom sklearn.datasets import make_classification\nfrom sklearn.ensemble import RandomForestClassifier\n\nfrom SEP_CFE_DiCE import (\n    FeatureRangeConstraint,\n    OrderedFeaturesConstraint,\n    FeatureThresholdConstraint,\n    build_constrained_explainer,\n    generate_counterfactuals,\n    extract_first_counterfactual_df,\n    plot_counterfactual_deltas,\n    plot_counterfactual_profiles,\n)

## 3) Create synthetic training/query data

In [ ]:
X, y = make_classification(\n    n_samples=600,\n    n_features=6,\n    n_informative=5,\n    n_redundant=0,\n    random_state=42,\n)\n\nfeature_names = [f"f{i}" for i in range(X.shape[1])]\ndf = pd.DataFrame(X, columns=feature_names)\ndf["Event_Y_N"] = y.astype(int)\n\ntrain_df = df.iloc[:500].reset_index(drop=True)\nquery_df = df.iloc[500:501].drop(columns=["Event_Y_N"]).reset_index(drop=True)\n\ntrain_df.head()

## 4) Train a model

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42)\nmodel.fit(train_df.drop(columns=["Event_Y_N"]), train_df["Event_Y_N"])

## 5) Define constraints

In [ ]:
permitted_ranges = {\n    col: [float(train_df[col].quantile(0.01)), float(train_df[col].quantile(0.99))]\n    for col in feature_names\n}\n\nconstraints = [\n    FeatureRangeConstraint(ranges=permitted_ranges, penalty_value=500.0),\n    OrderedFeaturesConstraint(\n        ordered_feature_groups=[("f0", "f1", "f2")],\n        increasing=False,\n        strict=False,\n        penalty_value=50.0,\n    ),\n    FeatureThresholdConstraint(\n        feature_name="f0",\n        upper_bound=float(train_df["f0"].quantile(0.95)),\n        penalty_value=100.0,\n    ),\n]\n\nconstraints

## 6) Build constrained explainer

In [ ]:
explainer, interfaces = build_constrained_explainer(\n    dataframe=train_df,\n    model=model,\n    outcome_name="Event_Y_N",\n    constraints=constraints,\n    l0_penalty_weight=0.2,\n)

## 7) Generate counterfactuals\n\nDepending on your machine this can take a little while.

In [ ]:
cf_obj = generate_counterfactuals(\n    explainer=explainer,\n    query_instances=query_df,\n    total_cfs=3,\n    desired_class=1,\n    maxiterations=300,\n    population_size=40,\n    proximity_weight=0.2,\n    sparsity_weight=0.2,\n    diversity_weight=2.0,\n)\n\ncf_df = extract_first_counterfactual_df(cf_obj)\ncf_df

## 8) Visualize feature deltas and profiles

In [ ]:
feature_cf_df = cf_df.drop(columns=["Event_Y_N"], errors="ignore")\n_ = plot_counterfactual_deltas(query_df.iloc[[0]], feature_cf_df, top_k=6)\n_ = plot_counterfactual_profiles(query_df.iloc[[0]], feature_cf_df, top_k=6, max_counterfactuals=3)